### Main Feature Extractor2

In [18]:
# feature_extractor2.py
import os
import json
import torch
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

# RDKit fingerprints
from rdkit.Chem import AllChem
from rdkit.Avalon import pyAvalonTools
from rdkit.Chem.rdReducedGraphs import GetErGFingerprint
from rdkit.DataStructs import ConvertToNumpyArray
from rdkit.Chem import MACCSkeys
from rdkit.Chem import rdFingerprintGenerator

# SELFIES
import selfies as sf
import torch.nn as nn

# -------------------------------
# Function to return chosen descriptors
# -------------------------------
def get_chosen_descriptors():
    chosen_descriptors = [
        'BalabanJ', 'BertzCT', 'Chi0', 'Chi0n', 'Chi0v', 'Chi1',
        'Chi1n', 'Chi1v', 'Chi2n', 'Chi2v', 'Chi3n', 'Chi3v', 'Chi4n', 'Chi4v',
        'EState_VSA1', 'EState_VSA10', 'EState_VSA11', 'EState_VSA2', 'EState_VSA3',
        'EState_VSA4', 'EState_VSA5', 'EState_VSA6', 'EState_VSA7', 'EState_VSA8',
        'EState_VSA9', 'ExactMolWt', 'FpDensityMorgan1', 'FpDensityMorgan2',
        'FpDensityMorgan3', 'FractionCSP3', 'HallKierAlpha', 'HeavyAtomCount',
        'HeavyAtomMolWt', 'Ipc', 'Kappa1', 'Kappa2', 'Kappa3', 'LabuteASA',
        'MaxAbsEStateIndex', 'MaxAbsPartialCharge', 'MaxEStateIndex', 'MaxPartialCharge',
        'MinAbsEStateIndex', 'MinAbsPartialCharge', 'MinEStateIndex', 'MinPartialCharge',
        'MolLogP', 'MolMR', 'MolWt', 'NHOHCount', 'NOCount', 'NumAliphaticCarbocycles',
        'NumAliphaticHeterocycles', 'NumAliphaticRings', 'NumAromaticCarbocycles',
        'NumAromaticHeterocycles', 'NumAromaticRings', 'NumHAcceptors', 'NumHDonors',
        'NumHeteroatoms', 'NumRadicalElectrons', 'NumRotatableBonds',
        'NumSaturatedCarbocycles', 'NumSaturatedHeterocycles', 'NumSaturatedRings',
        'NumValenceElectrons', 'PEOE_VSA1', 'PEOE_VSA10', 'PEOE_VSA11', 'PEOE_VSA12',
        'PEOE_VSA13', 'PEOE_VSA14', 'PEOE_VSA2', 'PEOE_VSA3', 'PEOE_VSA4', 'PEOE_VSA5',
        'PEOE_VSA6', 'PEOE_VSA7', 'PEOE_VSA8', 'PEOE_VSA9', 'RingCount', 'SMR_VSA1',
        'SMR_VSA10', 'SMR_VSA2', 'SMR_VSA3', 'SMR_VSA4', 'SMR_VSA5', 'SMR_VSA6', 'SMR_VSA7',
        'SMR_VSA8', 'SMR_VSA9', 'SlogP_VSA1', 'SlogP_VSA10', 'SlogP_VSA11', 'SlogP_VSA12',
        'SlogP_VSA2', 'SlogP_VSA3', 'SlogP_VSA4', 'SlogP_VSA5', 'SlogP_VSA6', 'SlogP_VSA7',
        'SlogP_VSA8', 'SlogP_VSA9', 'TPSA', 'VSA_EState1', 'VSA_EState10', 'VSA_EState2',
        'VSA_EState3', 'VSA_EState4', 'VSA_EState5', 'VSA_EState6', 'VSA_EState7',
        'VSA_EState8', 'VSA_EState9', 'fr_Al_COO', 'fr_Al_OH', 'fr_Al_OH_noTert', 'fr_ArN',
        'fr_Ar_COO', 'fr_Ar_N', 'fr_Ar_NH', 'fr_Ar_OH', 'fr_COO', 'fr_COO2', 'fr_C_O',
        'fr_C_O_noCOO', 'fr_C_S', 'fr_HOCCN', 'fr_Imine', 'fr_NH0', 'fr_NH1', 'fr_NH2',
        'fr_N_O', 'fr_Ndealkylation1', 'fr_Ndealkylation2', 'fr_Nhpyrrole', 'fr_SH',
        'fr_aldehyde', 'fr_alkyl_carbamate', 'fr_alkyl_halide', 'fr_allylic_oxid',
        'fr_amide', 'fr_amidine', 'fr_aniline', 'fr_aryl_methyl', 'fr_azide', 'fr_azo',
        'fr_barbitur', 'fr_benzene', 'fr_benzodiazepine', 'fr_bicyclic', 'fr_diazo',
        'fr_dihydropyridine', 'fr_epoxide', 'fr_ester', 'fr_ether', 'fr_furan', 'fr_guanido',
        'fr_halogen', 'fr_hdrzine', 'fr_hdrzone', 'fr_imidazole', 'fr_imide', 'fr_isocyan',
        'fr_isothiocyan', 'fr_ketone', 'fr_ketone_Topliss', 'fr_lactam', 'fr_lactone',
        'fr_methoxy', 'fr_morpholine', 'fr_nitrile', 'fr_nitro', 'fr_nitro_arom',
        'fr_nitro_arom_nonortho', 'fr_nitroso', 'fr_oxazole', 'fr_oxime',
        'fr_para_hydroxylation', 'fr_phenol', 'fr_phenol_noOrthoHbond', 'fr_phos_acid',
        'fr_phos_ester', 'fr_piperdine', 'fr_piperzine', 'fr_priamide', 'fr_prisulfonamd',
        'fr_pyridine', 'fr_quatN', 'fr_sulfide', 'fr_sulfonamd', 'fr_sulfone',
        'fr_term_acetylene', 'fr_tetrazole', 'fr_tameszole', 'fr_thiocyan', 'fr_thiophene',
        'fr_unbrch_alkane', 'fr_urea', 'qed'
    ]
    return chosen_descriptors

# -------------------------------
# SELFIES Embedder
# -------------------------------
class SelfiesEmbedder(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, output_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.fc = nn.Linear(emb_dim, output_dim)

    def forward(self, x):
        emb = self.embedding(x)          # (batch, seq, emb_dim)
        pooled = emb.mean(dim=1)         # mean pooling
        return self.fc(pooled)           # (batch, output_dim)

# -------------------------------
# Feature Extractor
# -------------------------------
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self,
                 selfies_vocab_path="selfies_vocab.json",
                 selfies_pad_len=300,
                 selfies_use_tfidf=True,
                 selfies_use_embed=True,
                 selfies_emb_dim=32,
                 selfies_emb_out=32,
                 morgan_bits=1024,
                 avalon_bits=1024,
                 erg_len=315,
                 maccs_bits=167,
                 rdkit_desc_nppb=None,
                 use_morgan=True,
                 use_avalon=True,
                 use_erg=True,
                 use_rdkit=True,
                 use_selfies=True,
                 use_maccs=True):
        # -------------------------------
        # Configuration
        # -------------------------------
        self.selfies_vocab_path = selfies_vocab_path
        self.selfies_pad_len = selfies_pad_len
        self.selfies_use_tfidf = selfies_use_tfidf
        self.selfies_use_embed = selfies_use_embed
        self.selfies_emb_dim = selfies_emb_dim
        self.selfies_emb_out = selfies_emb_out
        self.morgan_bits = morgan_bits
        self.avalon_bits = avalon_bits
        self.erg_len = erg_len
        self.maccs_bits = maccs_bits
        self.use_morgan = use_morgan
        self.use_avalon = use_avalon
        self.use_erg = use_erg
        self.use_rdkit = use_rdkit
        self.use_selfies = use_selfies
        self.use_maccs = use_maccs

        # Default RDKit descriptors
        if rdkit_desc_nppb is None:
            self.rdkit_desc_nppb = [name for name, _ in Descriptors._descList]
        else:
            self.rdkit_desc_nppb = rdkit_desc_nppb

        # Load vocab if exists
        self.selfies_vocab_stoi = None
        if os.path.exists(self.selfies_vocab_path):
            self.selfies_vocab_stoi = self._load_vocab()

        # TF-IDF
        self.vectorizer = None

        # Embedding
        self.token2idx = None
        self.model = None

    # -------------------------------
    # Get feature lengths by type
    # -------------------------------
    def get_feature_lengths(self):
        lengths = {}
        if self.use_morgan:
            lengths["Morgan"] = self.morgan_bits
        if self.use_avalon:
            lengths["Avalon"] = self.avalon_bits
        if self.use_erg:
            lengths["ErG"] = self.erg_len
        if self.use_rdkit:
            lengths["RDKit"] = len(self.rdkit_desc_nppb)
        if self.use_selfies:
            lengths["SELFIES_int"] = self.selfies_pad_len
            
        if self.selfies_use_tfidf and self.vectorizer:
            lengths["SELFIES_TFIDF"] = len(self.vectorizer.get_feature_names_out())
        if self.selfies_use_embed and self.model:
            lengths["SELFIES_Embed"] = self.selfies_emb_out
        if getattr(self, "use_maccs", False):
            lengths["MACCS"] = self.maccs_bits
        return lengths

    # -------------------------------
    # Build SELFIES vocab
    # -------------------------------
    def build_vocab(self, smiles_list):
        import json
        import selfies as sf
        from sklearn.feature_extraction.text import TfidfVectorizer
    
        selfies_set = set()
        selfies_texts = []
        token_lengths = []
    
        # Collect valid SMILES
        valid_smiles = []
        for smi in smiles_list:
            if not smi or not isinstance(smi, str):
                continue
            try:
                sf_str = sf.encoder(smi)
                tokens = list(sf.split_selfies(sf_str))  # <-- Convert generator to list
                selfies_set.update(tokens)
                token_lengths.append(len(tokens))
                if self.selfies_use_tfidf:
                    selfies_texts.append(" ".join(tokens))
                valid_smiles.append(smi)
            except Exception:
                continue
    
        if len(valid_smiles) == 0:
            raise ValueError("No valid SMILES found! Cannot build SELFIES vocab.")

        # Determine max length based on 99th percentile
        max_len_99 = int(np.percentile(token_lengths, 99))
        self.max_selfies_len = max_len_99  # store in the class
        self.selfies_pad_len = max_len_99
    
        # --- Original vocab ---
        vocab = {tok: idx for idx, tok in enumerate(sorted(selfies_set))}
        if hasattr(self, "selfies_vocab_path") and self.selfies_vocab_path:
            with open(self.selfies_vocab_path, "w") as f:
                json.dump(vocab, f, indent=4)
        self.selfies_vocab_stoi = vocab
    
        # --- Initialize TF-IDF ---
        if self.selfies_use_tfidf and len(selfies_texts) > 0:
            self.vectorizer = TfidfVectorizer(
                analyzer="word",
                token_pattern=r"\[.*?\]",
                lowercase=False,
                max_features=max_len_99  # limit size to 99th percentile
            )
            self.vectorizer.fit(selfies_texts)
        else:
            self.vectorizer = None  # skip TF-IDF if no valid texts
    
        # --- Initialize embeddings ---
        if getattr(self, "selfies_use_embed", False) and len(selfies_set) > 0:
            self.token2idx = {tok: i + 1 for i, tok in enumerate(sorted(selfies_set))}
            vocab_size = len(self.token2idx) + 1
            self.model = SelfiesEmbedder(
                vocab_size=vocab_size,
                emb_dim=self.selfies_emb_dim,
                output_dim=self.selfies_emb_out
            )
            self.model.eval()
    
        return vocab

    # -------------------------------
    # Load vocab
    # -------------------------------
    def _load_vocab(self):
        with open(self.selfies_vocab_path, "r") as f:
            return json.load(f)

    # -------------------------------
    # Transform SMILES -> features
    # -------------------------------
    def transform(self, smiles_list):
        # morgan_gen = lambda mol: AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=self.morgan_bits)
        # make generator once
        morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
            radius=2, 
            fpSize=self.morgan_bits
        )
        
        # correct way to compute bit vect
        morgan_gen = lambda mol: morgan_generator.GetFingerprint(mol)
        rows = []

        for smi in smiles_list:
            mol = self._smiles_to_mol(smi)
            parts = []

            # -------------------------------
            # Morgan
            # -------------------------------
            if self.use_morgan:
                if mol:
                    mfp = morgan_gen(mol)
                    arr = np.zeros((self.morgan_bits,), dtype=np.int8)
                    ConvertToNumpyArray(mfp, arr)
                else:
                    arr = np.zeros((self.morgan_bits,), dtype=np.int8)
                parts.append(arr.astype(np.float32))

            # -------------------------------
            # Avalon
            # -------------------------------
            if self.use_avalon:
                if mol:
                    try:
                        afp = pyAvalonTools.GetAvalonFP(mol, nBits=self.avalon_bits)
                        a_arr = np.zeros((self.avalon_bits,), dtype=np.int8)
                        ConvertToNumpyArray(afp, a_arr)
                    except Exception:
                        a_arr = np.zeros((self.avalon_bits,), dtype=np.int8)
                else:
                    a_arr = np.zeros((self.avalon_bits,), dtype=np.int8)
                parts.append(a_arr.astype(np.float32))

            # -------------------------------
            # ErG
            # -------------------------------
            if self.use_erg:
                if mol:
                    try:
                        erg_fp = GetErGFingerprint(mol)
                        erg_arr = np.asarray(list(erg_fp), dtype=np.float32)
                        if len(erg_arr) < self.erg_len:
                            erg_arr = np.pad(erg_arr, (0, self.erg_len - len(erg_arr)))
                        else:
                            erg_arr = erg_arr[:self.erg_len]
                    except Exception:
                        erg_arr = np.zeros((self.erg_len,), dtype=np.float32)
                else:
                    erg_arr = np.zeros((self.erg_len,), dtype=np.float32)
                parts.append(erg_arr.astype(np.float32))

            # -------------------------------
            # RDKit descriptors
            # -------------------------------
            if self.use_rdkit:
                if mol:
                    desc_arr = self._compute_descriptors(mol)
                else:
                    desc_arr = np.zeros(len(self.rdkit_desc_nppb), dtype=np.float32)
                parts.append(desc_arr.astype(np.float32))

            # -------------------------------
            # SELFIES integer
            # -------------------------------
            if self.use_selfies and self.selfies_vocab_stoi is not None:
                try:
                    sf_str = sf.encoder(smi)
                    ids = [self.selfies_vocab_stoi.get(t, 0) for t in sf.split_selfies(sf_str)]
                    if len(ids) < self.selfies_pad_len:
                        ids += [0]*(self.selfies_pad_len - len(ids))
                    else:
                        ids = ids[:self.selfies_pad_len]
                    parts.append(np.array(ids, dtype=np.int16).astype(np.float32))
                except Exception:
                    parts.append(np.zeros(self.selfies_pad_len, dtype=np.float32))

            # -------------------------------
            # SELFIES TF-IDF
            # -------------------------------
            if self.selfies_use_tfidf and self.vectorizer:
                try:
                    toks = sf.split_selfies(sf.encoder(smi))
                    text = " ".join(toks)
                    tfidf_vec = self.vectorizer.transform([text]).toarray().flatten()
                except Exception:
                    tfidf_vec = np.zeros(len(self.vectorizer.get_feature_names_out()), dtype=np.float32)
                parts.append(tfidf_vec.astype(np.float32))

            # -------------------------------
            # SELFIES Embeddings
            # -------------------------------
            if self.selfies_use_embed and self.model and self.token2idx:
                try:
                    toks = sf.split_selfies(sf.encoder(smi))
                    ids = [self.token2idx.get(t,0) for t in toks]
                    if len(ids) < self.selfies_pad_len:
                        ids += [0]*(self.selfies_pad_len - len(ids))
                    else:
                        ids = ids[:self.selfies_pad_len]
                    ids_tensor = torch.tensor([ids])
                    with torch.no_grad():
                        emb = self.model(ids_tensor).numpy().flatten()
                    parts.append(emb.astype(np.float32))
                except Exception:
                    parts.append(np.zeros(self.selfies_emb_out, dtype=np.float32))

            # -------------------------------
            # MACCS Keys
            # -------------------------------
            if getattr(self, "use_maccs", False):
                if mol:
                    try:
                        maccs_fp = MACCSkeys.GenMACCSKeys(mol)  # RDKit returns 167 bits
                        maccs_arr = np.zeros((self.maccs_bits,), dtype=np.int8)
                        ConvertToNumpyArray(maccs_fp, maccs_arr)
                    except Exception:
                        # fallback to zeros if RDKit fails
                        maccs_arr = np.zeros((self.maccs_bits,), dtype=np.int8)
                else:
                    # no molecule → zeros
                    maccs_arr = np.zeros((self.maccs_bits,), dtype=np.int8)
            
                # Ensure always same length
                if maccs_arr.shape[0] != self.maccs_bits:
                    raise ValueError(f"MACCS length mismatch: expected {self.maccs_bits}, got {maccs_arr.shape[0]}")
            
                parts.append(maccs_arr.astype(np.float32))
            
            # Final row = concat all selected parts
            row = np.concatenate(parts)
            rows.append(row)
            
        return np.vstack(rows)
        
    # -------------------------------
    # Helpers
    # -------------------------------
    def _smiles_to_mol(self, smi):
        try:
            return Chem.MolFromSmiles(smi)
        except:
            return None

    def _compute_descriptors(self, mol):
        values = []
        for name in self.rdkit_desc_nppb:
            try:
                fn = getattr(Descriptors, name)
                val = fn(mol)
                if np.isnan(val) or np.isinf(val):
                    val = 0.0
                values.append(float(val))
            except:
                values.append(0.0)
        return np.array(values, dtype=np.float32)

    def save_vocab_csv(self, csv_path="selfies_vocab.csv"):
        if self.selfies_vocab_stoi is None:
            raise ValueError("Vocab not built yet.")
        df = pd.DataFrame(list(self.selfies_vocab_stoi.items()), columns=["Token", "Index"])
        df.to_csv(csv_path, index=False)
        print(f"SELFIES vocab saved to {csv_path}")


In [23]:
import os
import pandas as pd
import numpy as np
import selfies as sf
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from rdkit import RDLogger

# Disable RDKit warnings
RDLogger.DisableLog('rdApp.*')

# ---------------------------
# Input and output setup
# ---------------------------
file_path = ""
output_dir = ""
os.makedirs(output_dir, exist_ok=True)

# ---------------------------
# Load dataset
# ---------------------------
df_pgp = pd.read_csv(file_path, encoding="utf-8-sig")
# === Rename the column ===
df_pgp.rename(columns={"Activity": "Label"}, inplace=True)
# === Save back to the same file (overwrite) ===
df_pgp.to_csv(file_path, index=False, encoding="utf-8-sig")

print(f"Loaded dataset with shape: {df_pgp.shape}")
print(f"Columns: {list(df_pgp.columns)}")

if 'Canonical_SMILES' not in df_pgp.columns or 'Label' not in df_pgp.columns:
    raise ValueError("Dataset must have columns 'Canonical_SMILES' and 'Label'")

smiles_list = df_pgp['Canonical_SMILES'].tolist()
labels = df_pgp['Label'].tolist()

# ---------------------------
# Initialize FeatureExtractor
# ---------------------------
fe = FeatureExtractor(
    use_morgan=True,
    use_avalon=True,
    use_erg=True,
    
    use_selfies=False,
    use_rdkit=True,
    use_maccs=True,
    selfies_use_tfidf=False,
    selfies_use_embed=False
)

# Build vocab using all SMILES
fe.build_vocab(smiles_list)

# Save vocab
fe.save_vocab_csv(os.path.join(output_dir, "pgp_vocab.csv"))

# Log feature lengths
feature_lengths = fe.get_feature_lengths()
log_lines = ["Feature lengths by method:"]
for method, length in feature_lengths.items():
    log_lines.append(f"  {method}: {length}")

# ---------------------------
# Extract & Save Raw Features
# ---------------------------
print("Extracting molecular features...")
X = fe.transform(smiles_list)
X_df = pd.DataFrame(X)
X_df.insert(0, "Canonical_SMILES", smiles_list)
X_df.insert(1, "Label", labels)

raw_path = os.path.join(output_dir, "pgp_fp_features_raw.csv")
X_df.to_csv(raw_path, index=False)
log_lines.append(f"Raw features saved: {raw_path}")
log_lines.append(f"Feature matrix shape: {X_df.shape}")

# ---------------------------
# Generate & Save SELFIES
# ---------------------------
print("Encoding SELFIES...")
selfies_list = []
for smi in smiles_list:
    try:
        selfies_list.append(sf.encoder(smi))
    except sf.EncoderError:
        selfies_list.append("INVALID_SMILES")

selfies_df = pd.DataFrame({"Canonical_SMILES": smiles_list, "SELFIES": selfies_list})
selfies_path = os.path.join(output_dir, "pgp_selfies.csv")
selfies_df.to_csv(selfies_path, index=False)
log_lines.append("SELFIES saved.")

# ---------------------------
# Impute + Scale
# ---------------------------
print("Performing imputation and scaling...")
X_fp = X_df.drop(columns=["Canonical_SMILES", "Label"])

# Replace inf/-inf with NaN
X_fp.replace([np.inf, -np.inf], np.nan, inplace=True)

# --- Imputation ---
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X_fp)
X_imputed_df = pd.DataFrame(X_imputed, columns=X_fp.columns)
X_imputed_df.to_csv(os.path.join(output_dir, "pgp_fp_features_imputed.csv"), index=False)
log_lines.append("Imputation done (NaNs replaced with column mean).")

# --- Scaling ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)
X_scaled_df = pd.DataFrame(X_scaled, columns=X_fp.columns)
X_scaled_df.to_csv(os.path.join(output_dir, "pgp_fp_features_scaled.csv"), index=False)
log_lines.append("Scaling done (StandardScaler).")

# ---------------------------
# Save log
# ---------------------------
log_path = os.path.join(output_dir, "pgp_preprocessing_log.txt")
with open(log_path, "w") as f:
    f.write("\n".join(log_lines))

print(f"✅ Preprocessing complete! All files saved in: {output_dir}")


Loaded dataset with shape: (528, 4)
Columns: ['SMILES', 'Label', 'Canonical_SMILES', 'Tanimoto_Similarity']
SELFIES vocab saved to E:\K-MELLODDY-Project\data\CADD-SC ADMET_Prediction_Models main data\AutoML_test_set\AutoML_canonical_SMILES\canonical_test_set\pgp_AutoML\5Folds\pgp_vocab.csv
Extracting molecular features...
Encoding SELFIES...
Performing imputation and scaling...
✅ Preprocessing complete! All files saved in: E:\K-MELLODDY-Project\data\CADD-SC ADMET_Prediction_Models main data\AutoML_test_set\AutoML_canonical_SMILES\canonical_test_set\pgp_AutoML\5Folds
